In [1]:
from common_functions_h01 import get_info_from_bigquery, fix_layer_mem, get_base2agglo
import json
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Load cells info from "gs://h01-release/data/20210601/c3/segment_properties/info"

In [2]:
# Load info from the JSON file
with open('info.json', 'r') as f:
    info_dict = json.load(f)

id_tags = {}
for id in range(0,46637):
  for ii in info_dict['inline']['properties']:
    try:
      try:
        id_tags[int(float(info_dict['inline']['ids'][id]))] = ii['tags'][ii['values'][id][0]]+ii['tags'][ii['values'][id][1]]+ii['tags'][ii['values'][id][2]]+ii['tags'][ii['values'][id][3]]        
      except:
        id_tags[int(float(info_dict['inline']['ids'][id]))] = ii['tags'][ii['values'][id][0]]+ii['tags'][ii['values'][id][1]]+ii['tags'][ii['values'][id][2]]        
    except:
      try:
        id_tags[int(float(info_dict['inline']['ids'][id]))] = ii['tags'][ii['values'][id][0]]+ii['tags'][ii['values'][id][1]]        
      except:
        try:
            id_tags[int(float(info_dict['inline']['ids'][id]))] = ii['tags'][ii['values'][id][0]]            
        except:
            break
len(id_tags), id_tags[3195297945]

(46637, 'L4pyramidalneuron')


## Load cell positions from "h01 github"

In [3]:
cell_ids = 'agglo_20200916c3_cell_data.json'

with open(cell_ids, 'r') as fp:
    all_cell_data = json.load(fp)

all_neurons = [int(float(x['agglo_seg'])) for x in all_cell_data if 'neuron' in x['type']]
all_cells = [int(float(x['agglo_seg'])) for x in all_cell_data]

print("Cells_agglo: %d, Neurons_agglo: %d, \nCells_info: %d, Neuron_info: %d, \nUnique: %d" % (len(all_cells), len(all_neurons), len(id_tags), len([type for type in id_tags.values() if 'neuron' in type]), len(id_tags.keys() | all_cells)))

all_neurons = set([int(float(x['agglo_seg'])) for x in all_cell_data if 'neuron' in x['type']])
all_cells = set([int(float(x['agglo_seg']))  for x in all_cell_data])


print("\n\nCells_agglo: %d, Neurons_agglo: %d, \nCells_info: %d, Neuron_info: %d, \nUnique: %d" % (len(all_cells), len(all_neurons), len(id_tags), len([type for type in id_tags.values() if 'neuron' in type]), len(set(id_tags.keys()) | all_cells)))

Cells_agglo: 49377, Neurons_agglo: 16067, 
Cells_info: 46637, Neuron_info: 15567, 
Unique: 46919


Cells_agglo: 45994, Neurons_agglo: 15686, 
Cells_info: 46637, Neuron_info: 15567, 
Unique: 46919


## incluing cells 925 neural positions not inclued in h01 github

In [4]:
ids_List_neurons = sorted([id for id in all_cells])
neuron_info_List = [id for id in id_tags.keys() if id not in ids_List_neurons]
len((neuron_info_List)), len(ids_List_neurons)

(925, 45994)

In [5]:
# data from c3_cloudvolume = cloudvolume.CloudVolume('gs://h01-release/data/20210601/c3', progress=True)
ind_cell_df = pd.read_csv('cell_positions_h01_925_missing.csv') # missing positions from sk[i].nodes['radius'].idxmax()
ind_cell_df

,agglo_seg,x,y,z
0,782058771,2966751,452127,3828
1,913205129,3005951,492895,57056
2,1216687103,2497087,612447,50985
3,1375838707,2218176,558367,49598
4,1855013837,1884384,751392,15741
...,...,...,...,...
920,105659742627,1312192,2293887,163977
921,105671572495,659392,2298496,171138
922,105672884077,974687,2297216,167606
923,105673920419,1223904,2297535,169289


In [6]:
id_and_xy = [[int(float(x['agglo_seg']))+i*1000000000000, int(x['true_x']), int(x['true_y'])] for i,x in enumerate(all_cell_data)]
print(np.shape(id_and_xy))

for id in list(ind_cell_df['agglo_seg'].values):
    id_and_xy.append([id, ind_cell_df[ind_cell_df['agglo_seg'] == id]['x'].values[0], ind_cell_df[ind_cell_df['agglo_seg'] == id]['y'].values[0]])

print(np.shape(id_and_xy))

(49377, 3)
(50302, 3)


In [7]:
layer_bounds = 'cortical_bounds_circles.json'

with open(layer_bounds, "r") as f:
    bounds = json.load(f)

old_to_new_format_naming = {'Layer 1': 'layer_1', 'Layer 2': 'layer_2', 'Layer 3': 'layer_3', 'Layer 4': 'layer_4', 'Layer 5': 'layer_5', 'Layer 6': 'layer_6', 'White matter': 'layer_7'}
new_to_old_format_naming = {v: k for k, v in old_to_new_format_naming.items()}

id_and_xy = np.array(id_and_xy)

shard_layers = fix_layer_mem(bounds, id_and_xy)[0]

aggloid2layer = {}
for layer in shard_layers.keys():
    for seg_id in shard_layers[layer]:
        aggloid2layer[seg_id] = old_to_new_format_naming[layer]

print(len(aggloid2layer))

50302


In [8]:
ids_t = []
for i,x in enumerate(all_cell_data):
    if int(float(x['agglo_seg'])) not in id_tags.keys():
        # print(int(float(x['agglo_seg'])), "not in aggloid2layer")
        ids_t.append(int(float(x['agglo_seg'])))

ids_t = np.unique(ids_t)

for id in ids_t:
    id_tags[id] = x['type']

for i,x in enumerate(all_cell_data):
    if int(float(x['agglo_seg'])) in ids_t:
        id_tags[int(float(x['agglo_seg']))] = x['type']

In [9]:
def tag_layer(tag):
    c = tag[1:2]
    return int(c) if c.isdigit() else 0

id_and_xyz = [[int(float(x['agglo_seg'])), 
               int(x['true_x']), 
               int(x['true_y']), 
               int(x['true_z']),
               x['type'] , 
               int(aggloid2layer[int(x['agglo_seg'])+i*1000000000000][-1:]), 
               tag_layer(id_tags[int(float(x['agglo_seg']))]) if tag_layer(id_tags[int(float(x['agglo_seg']))]) > 0 else int(aggloid2layer[int(x['agglo_seg'])+i*1000000000000][-1:]), 
               id_tags[int(float(x['agglo_seg']))],
               "neuron" in id_tags[int(float(x['agglo_seg']))]] for i,x in enumerate(all_cell_data)]

type_new = {}
type_new['layer-unclassifiedglialoligodendrocyte']='oligodendrocyte'
type_new['layer-unclassifiedglialmicroglia/opc']='microglia/opc'
type_new['layer-unclassifiedpyramidalneuron']='pyramidal neuron'
type_new['layer-unclassifiedastrocyteglial']='astrocyte'
type_new['layer-unclassifiedneuronunclassified-neuron']='unclassified neuron'
type_new['layer-unclassifiedneuroninterneuron']='interneuron'
type_new['layer-unclassifiedneuronexcitatory/spiny-with-atypical-tree']='excitatory/spiny neuron with atypical tree'
type_new['layer-unclassifiedunknown-typedark']='unknown cell'
type_new['layer-unclassifiedunknown-type']='unknown cell'
type_new['layer-unclassifiedblood-vessel-cell']='blood vessel cell'
type_new['layer-unclassifiedpyramidalneuronbipolar']='pyramidal neuron'
type_new['layer-unclassifiedunknown-typegrey']='unknown cell'
type_new['layer-unclassifiedneuronexcitatory/spiny-with-atypical-treethin-apical-dendrite']='excitatory/spiny neuron with atypical tree'
type_new['layer-unclassifiedpyramidalneuronsparsely-spiny']='pyramidal neuron'
type_new['layer-unclassifiedneuronunclassified-neuronthin-apical-dendrite']='unclassified neuron'
type_new['layer-unclassifiedneuroninterneuronbipolar']='interneuron'
type_new['layer-unclassifiedneuronspiny-stellate']='spiny stellate neuron'
type_new['layer-unclassifiedpyramidalneurondark']='pyramidal neuron'
type_new['layer-unclassifiedglialmicroglia/opcc-shaped']='microglia/opc'
type_new['layer-unclassifiedneuronexcitatory/spiny-with-atypical-treeinverted-apical-dendrite']='excitatory/spiny neuron with atypical tree'
type_new['layer-unclassifiedneuronexcitatory/spiny-with-atypical-treebipolar']='excitatory/spiny neuron with atypical tree'
type_new['layer-unclassifiedneuronunclassified-neuronpossible-interneuron']='unclassified neuron'

for id in sorted(ind_cell_df['agglo_seg'].values):
    id_and_xyz.append([id, 
                       ind_cell_df[ind_cell_df['agglo_seg'] == id]['x'].values[0], 
                       ind_cell_df[ind_cell_df['agglo_seg'] == id]['y'].values[0], 
                       ind_cell_df[ind_cell_df['agglo_seg'] == id]['z'].values[0],
                       type_new.get(id_tags[id], id_tags[id]),
                       int(aggloid2layer[id][-1:]),
                       tag_layer(id_tags[id]) if tag_layer(id_tags[id]) > 0 else int(aggloid2layer[id][-1:]),
                       id_tags[id],
                       "neuron" in id_tags[id]])

print(np.shape(id_and_xyz))

(50302, 9)


In [10]:
nodes_new = pd.DataFrame(id_and_xyz, columns=['agglo_seg','x', 'y', 'z', 'type', 'layer', 'linfo', 'info', 'neuron'])
# nodes_new = nodes_new.drop_duplicates(subset=['agglo_seg'], keep='first')
nodes_new["type"].value_counts()

oligodendrocyte                               20759
pyramidal neuron                               8807
microglia/opc                                  6595
astrocyte                                      5517
interneuron                                    4691
unclassified neuron                            1316
excitatory/spiny neuron with atypical tree     1224
unknown cell                                    771
blood vessel cell                               290
spiny stellate neuron                           186
c-shaped cell                                   146
Name: type, dtype: int64

In [11]:
nodes_new["layer"].value_counts()

7    14243
3     7853
2     7422
5     7133
4     5437
6     4499
1     3715
Name: layer, dtype: int64

In [12]:
nodes_new["linfo"].value_counts()

7    13908
2     7225
4     7097
5     7082
3     6960
6     4367
1     3663
Name: linfo, dtype: int64

In [13]:
e_i_ratios_df = pd.read_csv('goog14r0s5c3_e_i_ratios_feb_2023.csv') 
e_i_ratios_df

,segment id,excitatory synapses,inhibitory synapses,cortical layer,cell type
0,3622918632,7,27,Layer 1,astrocyte
1,16335235294,0,0,White matter,oligodendrocyte
2,69557313527,6,4,White matter,astrocyte
3,30595974593,504,208,Layer 2,interneuron
4,74128752202,0,0,Layer 5,oligodendrocyte
...,...,...,...,...,...
45989,86350168540,0,0,Layer 3,oligodendrocyte
45990,24806968514,0,0,White matter,oligodendrocyte
45991,96457698151,0,0,White matter,oligodendrocyte
45992,38391271379,1,9,Layer 4,pyramidal neuron


In [14]:
lists = {}
for id in e_i_ratios_df["segment id"].values:
    lists[id] = {}
    for key in e_i_ratios_df.keys():
        lists[id][key] = []

for id in e_i_ratios_df["segment id"].values:
    for key in e_i_ratios_df.keys():
        lists[id][key].append(e_i_ratios_df[e_i_ratios_df["segment id"] == id][key].values[0])

In [15]:
for key in e_i_ratios_df.keys()[1:]:
    nodes_new[key] = nodes_new["agglo_seg"].map(lambda x: lists[x][key][0] if x in lists else 0)

In [16]:
repetitions = []

nodes_new = nodes_new.sort_values(by='agglo_seg', ascending=True)

aa, bb = np.unique([nodes_new["agglo_seg"].values], return_counts=True)

aux = zip(aa, bb)
aux = dict(aux)

for i,id in enumerate(nodes_new["agglo_seg"].values):
        repetitions.append(aux[id])


In [17]:
nodes_new["synapses"] = nodes_new["excitatory synapses"] + nodes_new["inhibitory synapses"]
nodes_new["repetitions"] = repetitions
nodes_new["repetitions"].value_counts()

1       45583
2        2308
1095     1095
3         402
173       173
159       159
121       121
4         100
100       100
96         96
52         52
5          40
6          24
23         23
7          14
12         12
Name: repetitions, dtype: int64

In [18]:
set(nodes_new[nodes_new["repetitions"] > 12]["agglo_seg"].values)

{0,
 100000000000,
 101000000000,
 102000000000,
 103000000000,
 104000000000,
 105000000000,
 106000000000}

In [19]:
nodes_new = nodes_new[nodes_new["repetitions"] <= 12]
nodes_new["repetitions"].value_counts()

1     45583
2      2308
3       402
4       100
5        40
6        24
7        14
12       12
Name: repetitions, dtype: int64

In [20]:
nodes_new[nodes_new["neuron"] == False]["type"].value_counts()

oligodendrocyte                               19536
microglia/opc                                  6317
astrocyte                                      5388
unknown cell                                    683
blood vessel cell                               281
c-shaped cell                                   144
pyramidal neuron                                 87
interneuron                                      70
unclassified neuron                              44
excitatory/spiny neuron with atypical tree       11
spiny stellate neuron                             2
Name: type, dtype: int64

In [21]:
nodes_new["labels"] = nodes_new["type"].astype(str) + "_" + nodes_new["info"].astype(str) + "_" + nodes_new["cell type"].astype(str)
nodes_new["neuron"] = nodes_new["labels"].str.contains("neuron")
nodes_new["glia"] = nodes_new["labels"].str.contains("glia")
nodes_new["spiny"] = nodes_new["labels"].str.contains("spiny")
nodes_new["pyr"] = nodes_new["labels"].str.contains("pyr")
# nodes_new[(nodes_new["neuron"] == False) & (nodes_new["type"] == "unknown cell") & (nodes_new["glia"] == False) & (nodes_new["synapses"] > 100) & (nodes_new["layer"] != 1)]

In [22]:
# for id in set(nodes_new[(nodes_new["neuron"] == False) & (nodes_new["type"] == "unknown cell") & (nodes_new["glia"] == False) & (nodes_new["synapses"] > 100) & (nodes_new["layer"] != 1)]["agglo_seg"].values):
#     if id not in nodes_new[nodes_new["neuron"] == True]["agglo_seg"].values:
#         print(id,",")
nodes_new[(nodes_new["neuron"] == False) & (nodes_new["type"] == "unknown cell") & (nodes_new["glia"] == False) & (nodes_new["synapses"] > 100) & (nodes_new["layer"] != 1)]["neuron"].value_counts()

False    123
Name: neuron, dtype: int64

In [23]:
nodes_new[nodes_new["agglo_seg"] == 5755262032]

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr
29636,5755262032,1309952,1972360,138069,oligodendrocyte,7,6,L6glialoligodendrocyte,False,4265,1149,Layer 6,unknown cell,5414,3,oligodendrocyte_L6glialoligodendrocyte_unknown...,True,False,False
2843,5755262032,1487352,1781616,70485,pyramidal neuron,6,6,L6glialoligodendrocyte,True,4265,1149,Layer 6,unknown cell,5414,3,pyramidal neuron_L6glialoligodendrocyte_unknow...,True,False,True
29638,5755262032,1306288,1970640,129621,unknown cell,7,6,L6glialoligodendrocyte,False,4265,1149,Layer 6,unknown cell,5414,3,unknown cell_L6glialoligodendrocyte_unknown cell,True,False,False


In [24]:
neurons = nodes_new[(nodes_new["neuron"] == True) | (nodes_new["neuron"] == False) & (nodes_new["type"] == "unknown cell") & (nodes_new["glia"] == False) & (nodes_new["synapses"] > 100) & (nodes_new["layer"] != 1)]
# neurons["type"].value_counts()
neurons["neuron"].value_counts()

True     16203
False      123
Name: neuron, dtype: int64

In [25]:
# The "unknown cell" are neurons if synapses > 100,  ADDING 115 EXTRA CELLS!!!!
# but we can be more strict and say that they are neurons if synapses > 2000, 
# since some "unknown cell" with synapses between 100 and 2000 could be glia. Let's check the "unknown cell" with synapses > 2000

# known_neurons = neurons[neurons["type"] != "unknown cell"]
# known_neurons["type"].value_counts()

# unknown_neurons = neurons[(neurons["type"] == "unknown cell") & (neurons["neuron"] == False)] # some "unknown cell" are neurons. Create a rules to separate them
# unknown_neurons_withSyns = unknown_neurons[unknown_neurons["synapses"] > 2000]

# for id in set(unknown_neurons["agglo_seg"].values):
#     if id not in known_neurons["agglo_seg"].values:
#         print(id,",")
#         # print(id, id_tags[id])

# set(unknown_neurons_withSyns["agglo_seg"].values) # STILL HAVING SOME GLIAS IN THESE 115 CELLS. NEED TO CHECK THEM ONE BY ONE TO CREATE A BETTER RULE. 
# FOR EXAMPLE, CHECKING THE RATIO OF EXCITATORY/INHIBITORY SYNAPSES, SINCE GLIA HAVE MORE INHIBITORY SYNAPSES THAN EXCITATORY ONES. 
# ALSO CHECKING THE INFO OF THESE CELLS, SINCE SOME OF THEM HAVE INFO THAT SUGGESTS THEY ARE GLIA. 
# CHECK THE SKELETONS OF THESE CELLS TO SEE IF THEY HAVE A NEURONAL MORPHOLOGY OR NOT. NEURONS HAVE A MORE COMPLEX MORPHOLOGY THAN GLIA, WITH MORE BRANCHES AND SYNAPSES. 

In [26]:
neurons[neurons["agglo_seg"] == 16292920863]

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr
28596,16292920863,1108336,2059408,78933,oligodendrocyte,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,oligodendrocyte_WMneuronunclassified-neuronthi...,False,False,False
28591,16292920863,1101184,2048552,78933,oligodendrocyte,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,oligodendrocyte_WMneuronunclassified-neuronthi...,False,False,False
28587,16292920863,1101640,2050544,91605,unclassified neuron,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,unclassified neuron_WMneuronunclassified-neuro...,False,False,False
17911,16292920863,1234464,2056840,57813,oligodendrocyte,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,oligodendrocyte_WMneuronunclassified-neuronthi...,False,False,False
28555,16292920863,1139952,2026960,53760,oligodendrocyte,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,oligodendrocyte_WMneuronunclassified-neuronthi...,False,False,False
28592,16292920863,1096144,2051112,74709,oligodendrocyte,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,oligodendrocyte_WMneuronunclassified-neuronthi...,False,False,False
28593,16292920863,1108584,2055064,74709,oligodendrocyte,7,7,WMneuronunclassified-neuronthin-apical-dendrite,True,947,367,White matter,oligodendrocyte,1314,7,oligodendrocyte_WMneuronunclassified-neuronthi...,False,False,False


In [27]:
nodes_new["type"].value_counts()

oligodendrocyte                               19684
pyramidal neuron                               8715
microglia/opc                                  6375
astrocyte                                      5406
interneuron                                    4621
excitatory/spiny neuron with atypical tree     1202
unclassified neuron                            1157
unknown cell                                    708
blood vessel cell                               283
spiny stellate neuron                           186
c-shaped cell                                   146
Name: type, dtype: int64

In [28]:
ordering = ["pyramidal neuron","excitatory/spiny neuron with atypical tree","spiny stellate neuron",
        "interneuron",
        "unclassified neuron",
        "unknown cell",
        "oligodendrocyte","microglia/opc","astrocyte","blood vessel cell"]

order_map = {t: i for i, t in enumerate(ordering)}
neurons = neurons.sort_values(by="type",key=lambda x: x.map(order_map).fillna(len(ordering)))
neurons

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr
50292,105572531279,1336832,2275647,169884,pyramidal neuron,7,7,layer-unclassifiedpyramidalneuron,True,0,0,0,0,0,1,pyramidal neuron_layer-unclassifiedpyramidalne...,False,False,True
2924,6525902368,1523184,2008880,91605,pyramidal neuron,7,7,WMpyramidalneuron,True,2919,835,White matter,pyramidal neuron,3754,1,pyramidal neuron_WMpyramidalneuron_pyramidal n...,False,False,True
11087,6526033781,1565984,1840864,121173,pyramidal neuron,6,6,L6pyramidalneuronbipolar,True,2855,883,Layer 6,pyramidal neuron,3738,1,pyramidal neuron_L6pyramidalneuronbipolar_pyra...,False,False,True
7714,6527305613,2122232,1725400,108501,pyramidal neuron,5,5,L5pyramidalneuron,True,615,359,Layer 5,pyramidal neuron,974,1,pyramidal neuron_L5pyramidalneuron_pyramidal n...,False,False,True
12791,6527669458,1989968,1976000,100053,pyramidal neuron,5,6,L6pyramidalneuronbipolar,True,2603,840,Layer 5,pyramidal neuron,3443,1,pyramidal neuron_L6pyramidalneuronbipolar_pyra...,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2259,39278429512,1058600,1359080,146517,astrocyte,6,6,L6neuronexcitatory/spiny-with-atypical-tree,True,1105,501,Layer 6,excitatory/spiny neuron with atypical tree,1606,2,astrocyte_L6neuronexcitatory/spiny-with-atypic...,False,True,False
49332,33041949403,3385288,1897776,104277,blood vessel cell,2,2,L2pyramidalneuron,True,4417,1624,Layer 2,blood vessel cell,6041,3,blood vessel cell_L2pyramidalneuron_blood vess...,False,False,True
49333,33041949403,3380368,1901648,104277,blood vessel cell,2,2,L2pyramidalneuron,True,4417,1624,Layer 2,blood vessel cell,6041,3,blood vessel cell_L2pyramidalneuron_blood vess...,False,False,True
3418,29585541327,1671664,1003088,87381,c-shaped cell,4,4,L4neuroninterneuron,True,205,244,Layer 4,interneuron,449,2,c-shaped cell_L4neuroninterneuron_interneuron,False,False,False


In [29]:
# Fig. 4. Distribution of cells (Shapson-Coe, Science 2024)
# Spiny neurons (n = 10531; putatively excitatory) 
# Interneurons (n = 4688; few spines, putatively inhibitory)
# Astrocytes (n = 5474).
# Most of the oligodendrocytes (n = 20,139)
# Cell bodies (n = 6702) of microglia and OPCs.
neurons['type'].value_counts()

pyramidal neuron                              8715
interneuron                                   4621
excitatory/spiny neuron with atypical tree    1202
unclassified neuron                           1157
oligodendrocyte                                197
spiny stellate neuron                          186
unknown cell                                   149
microglia/opc                                   72
astrocyte                                       23
blood vessel cell                                2
c-shaped cell                                    2
Name: type, dtype: int64

In [30]:
INH_neurons = neurons[(neurons["type"] == "interneuron") | (neurons["spiny"] == False) & (neurons["pyr"] == False) | (neurons["layer"] == 1)]
INH_neurons = INH_neurons.drop_duplicates(subset=['agglo_seg'], keep='first')
INH_neurons["type"].value_counts()

interneuron            4616
unclassified neuron     666
unknown cell            133
microglia/opc             1
Name: type, dtype: int64

In [31]:
INH_neurons

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr
23838,46650883483,2780792,799384,138069,interneuron,2,2,L2neuroninterneuron,True,449,241,Layer 2,interneuron,690,1,interneuron_L2neuroninterneuron_interneuron,False,False,False
46950,55235394698,1739840,823720,125397,interneuron,3,3,L3neuroninterneuron,True,118,85,Layer 4,interneuron,203,1,interneuron_L3neuroninterneuron_interneuron,False,False,False
48478,43256937531,2246720,2166464,66261,interneuron,5,5,L5neuroninterneuron,True,2,4,Layer 5,interneuron,6,1,interneuron_L5neuroninterneuron_interneuron,False,False,False
40633,46405498324,3273000,754960,116949,interneuron,1,1,L1neuroninterneuron,True,326,476,Layer 1,interneuron,802,1,interneuron_L1neuroninterneuron_interneuron,False,False,False
24540,40320620268,3236272,1422056,112725,interneuron,2,2,L2neuroninterneuron,True,647,540,Layer 2,interneuron,1187,1,interneuron_L2neuroninterneuron_interneuron,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2539,22611620829,1501320,1407584,57813,unknown cell,5,5,L5unknown-type,False,261,149,Layer 5,unknown cell,410,1,unknown cell_L5unknown-type_unknown cell,False,False,False
23645,24798996438,2223624,2051256,66261,unknown cell,5,5,L5unknown-typegrey,False,421,211,Layer 5,unknown cell,632,2,unknown cell_L5unknown-typegrey_unknown cell,False,False,False
36102,1697644856,2505288,746104,19200,unknown cell,2,2,L2unknown-typegrey,False,172,122,Layer 3,unknown cell,294,1,unknown cell_L2unknown-typegrey_unknown cell,False,False,False
46453,34011994965,2039864,2090776,167637,unknown cell,5,6,L6unknown-typegrey,False,349,156,Layer 5,unknown cell,505,1,unknown cell_L6unknown-typegrey_unknown cell,False,False,False


In [32]:
spiny_neurons = neurons[(neurons["type"] == "pyramidal neuron") |  (neurons["type"] == "excitatory/spiny neuron with atypical tree") |  (neurons["type"] == "spiny stellate neuron") | (neurons["spiny"] == True) | (neurons["pyr"] == True)]
spiny_neurons = spiny_neurons.drop_duplicates(subset=['agglo_seg'], keep='first')
spiny_neurons = spiny_neurons[spiny_neurons["layer"] != 1]
spiny_neurons = spiny_neurons[spiny_neurons["type"] != "interneuron"]
spiny_neurons["type"].value_counts()

pyramidal neuron                              8706
excitatory/spiny neuron with atypical tree    1198
unclassified neuron                            484
spiny stellate neuron                          185
unknown cell                                     2
Name: type, dtype: int64

In [33]:
spiny_neurons

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr
50292,105572531279,1336832,2275647,169884,pyramidal neuron,7,7,layer-unclassifiedpyramidalneuron,True,0,0,0,0,0,1,pyramidal neuron_layer-unclassifiedpyramidalne...,False,False,True
2924,6525902368,1523184,2008880,91605,pyramidal neuron,7,7,WMpyramidalneuron,True,2919,835,White matter,pyramidal neuron,3754,1,pyramidal neuron_WMpyramidalneuron_pyramidal n...,False,False,True
11087,6526033781,1565984,1840864,121173,pyramidal neuron,6,6,L6pyramidalneuronbipolar,True,2855,883,Layer 6,pyramidal neuron,3738,1,pyramidal neuron_L6pyramidalneuronbipolar_pyra...,False,False,True
7714,6527305613,2122232,1725400,108501,pyramidal neuron,5,5,L5pyramidalneuron,True,615,359,Layer 5,pyramidal neuron,974,1,pyramidal neuron_L5pyramidalneuron_pyramidal n...,False,False,True
12791,6527669458,1989968,1976000,100053,pyramidal neuron,5,6,L6pyramidalneuronbipolar,True,2603,840,Layer 5,pyramidal neuron,3443,1,pyramidal neuron_L6pyramidalneuronbipolar_pyra...,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2651,39365947835,1143128,1183224,112725,unclassified neuron,5,5,L5neuronexcitatory/spiny-with-atypical-treespa...,True,144,97,Layer 5,unclassified neuron,241,1,unclassified neuron_L5neuronexcitatory/spiny-w...,False,True,False
8066,39355084243,2086096,1172032,167637,unclassified neuron,4,4,L4neuronexcitatory/spiny-with-atypical-treespa...,True,106,36,Layer 4,unclassified neuron,142,1,unclassified neuron_L4neuronexcitatory/spiny-w...,False,True,False
2505,4735805175,1631680,1457744,15360,unclassified neuron,5,5,L5neuronexcitatory/spiny-with-atypical-tree,True,713,381,Layer 5,unclassified neuron,1094,1,unclassified neuron_L5neuronexcitatory/spiny-w...,False,True,False
1793,32496366124,1618624,1727128,167637,unknown cell,6,6,L6neuronexcitatory/spiny-with-atypical-tree,True,323,104,Layer 6,unknown cell,427,1,unknown cell_L6neuronexcitatory/spiny-with-aty...,False,True,False


In [34]:
nodes_new

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr
39453,258279044,3018304,367192,26880,astrocyte,1,1,L1astrocyteglial,False,18,43,Layer 1,astrocyte,61,2,astrocyte_L1astrocyteglial_astrocyte,True,False,False
39878,258279044,3031784,347512,3840,astrocyte,1,1,L1astrocyteglial,False,18,43,Layer 1,astrocyte,61,2,astrocyte_L1astrocyteglial_astrocyte,True,False,False
35704,272206514,2888792,369680,42240,astrocyte,1,1,L1glialmicroglia/opc,False,15,23,Layer 1,astrocyte,38,1,astrocyte_L1glialmicroglia/opc_astrocyte,True,False,False
35703,272264889,2892640,366376,38400,astrocyte,1,1,L1astrocyteglial,False,10,23,Layer 1,astrocyte,33,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False
39460,272367447,2935032,361944,3840,astrocyte,1,1,L1astrocyteglial,False,10,27,Layer 1,astrocyte,37,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50297,105659742627,1312192,2293887,163977,microglia/opc,7,7,layer-unclassifiedglialmicroglia/opc,False,0,0,0,0,0,1,microglia/opc_layer-unclassifiedglialmicroglia...,True,False,False
50298,105671572495,659392,2298496,171138,oligodendrocyte,7,7,layer-unclassifiedglialoligodendrocyte,False,0,0,0,0,0,1,oligodendrocyte_layer-unclassifiedglialoligode...,True,False,False
50299,105672884077,974687,2297216,167606,oligodendrocyte,7,7,layer-unclassifiedglialoligodendrocyte,False,0,0,0,0,0,1,oligodendrocyte_layer-unclassifiedglialoligode...,True,False,False
50300,105673920419,1223904,2297535,169289,microglia/opc,7,7,layer-unclassifiedglialmicroglia/opc,False,0,0,0,0,0,1,microglia/opc_layer-unclassifiedglialmicroglia...,True,False,False


In [35]:
all = nodes_new["agglo_seg"].values
exc = sorted(set(spiny_neurons["agglo_seg"].values))
inh = sorted(set(INH_neurons["agglo_seg"].values))

# aux = zip(exc + inh, ["exc" for i in exc] + ["inh" for i in inh])
# aux = dict(aux)
aux = {}
for id in inh:
    aux[id] = "inh"
for id in exc:
    aux[id] = "exc"

In [36]:
EI = []

for i,id in enumerate(nodes_new["agglo_seg"].values):
        if id in aux.keys():
                EI.append(aux[id])
        else:
                EI.append("None")
nodes_new["EI"] = EI
nodes_new["EI"].value_counts()

None    31950
exc     10861
inh      5672
Name: EI, dtype: int64

In [37]:
EI = []

for i,id in enumerate(spiny_neurons["agglo_seg"].values):
        if id in aux.keys():
                EI.append(aux[id])
        else:
                EI.append("None")
spiny_neurons["EI"] = EI
spiny_neurons["EI"].value_counts()

exc    10575
Name: EI, dtype: int64

In [38]:
EI = []

for i,id in enumerate(INH_neurons["agglo_seg"].values):
        if id in aux.keys():
                EI.append(aux[id])
        else:
                EI.append("None")
INH_neurons["EI"] = EI
INH_neurons["EI"].value_counts()

inh    5394
exc      22
Name: EI, dtype: int64

In [39]:
INH_neurons = INH_neurons[INH_neurons["EI"] != "exc"]
INH_neurons["EI"].value_counts()

inh    5394
Name: EI, dtype: int64

In [40]:
neurons_new = pd.concat([spiny_neurons, INH_neurons], axis=0)
neurons_new

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr,EI
50292,105572531279,1336832,2275647,169884,pyramidal neuron,7,7,layer-unclassifiedpyramidalneuron,True,0,0,0,0,0,1,pyramidal neuron_layer-unclassifiedpyramidalne...,False,False,True,exc
2924,6525902368,1523184,2008880,91605,pyramidal neuron,7,7,WMpyramidalneuron,True,2919,835,White matter,pyramidal neuron,3754,1,pyramidal neuron_WMpyramidalneuron_pyramidal n...,False,False,True,exc
11087,6526033781,1565984,1840864,121173,pyramidal neuron,6,6,L6pyramidalneuronbipolar,True,2855,883,Layer 6,pyramidal neuron,3738,1,pyramidal neuron_L6pyramidalneuronbipolar_pyra...,False,False,True,exc
7714,6527305613,2122232,1725400,108501,pyramidal neuron,5,5,L5pyramidalneuron,True,615,359,Layer 5,pyramidal neuron,974,1,pyramidal neuron_L5pyramidalneuron_pyramidal n...,False,False,True,exc
12791,6527669458,1989968,1976000,100053,pyramidal neuron,5,6,L6pyramidalneuronbipolar,True,2603,840,Layer 5,pyramidal neuron,3443,1,pyramidal neuron_L6pyramidalneuronbipolar_pyra...,False,False,True,exc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5705,5344594921,961928,1605040,53760,unknown cell,7,7,WMunknown-type,False,571,290,White matter,unknown cell,861,1,unknown cell_WMunknown-type_unknown cell,False,False,False,inh
2539,22611620829,1501320,1407584,57813,unknown cell,5,5,L5unknown-type,False,261,149,Layer 5,unknown cell,410,1,unknown cell_L5unknown-type_unknown cell,False,False,False,inh
36102,1697644856,2505288,746104,19200,unknown cell,2,2,L2unknown-typegrey,False,172,122,Layer 3,unknown cell,294,1,unknown cell_L2unknown-typegrey_unknown cell,False,False,False,inh
46453,34011994965,2039864,2090776,167637,unknown cell,5,6,L6unknown-typegrey,False,349,156,Layer 5,unknown cell,505,1,unknown cell_L6unknown-typegrey_unknown cell,False,False,False,inh


In [41]:
neurons_new.to_csv('neurons_positions_h01_full.csv')

In [42]:
nodes_new[nodes_new["EI"] == "None"]["type"].value_counts()

oligodendrocyte      19359
microglia/opc         6255
astrocyte             5357
unknown cell           555
blood vessel cell      280
c-shaped cell          144
Name: type, dtype: int64

In [43]:
nodes_new

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr,EI
39453,258279044,3018304,367192,26880,astrocyte,1,1,L1astrocyteglial,False,18,43,Layer 1,astrocyte,61,2,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
39878,258279044,3031784,347512,3840,astrocyte,1,1,L1astrocyteglial,False,18,43,Layer 1,astrocyte,61,2,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
35704,272206514,2888792,369680,42240,astrocyte,1,1,L1glialmicroglia/opc,False,15,23,Layer 1,astrocyte,38,1,astrocyte_L1glialmicroglia/opc_astrocyte,True,False,False,None
35703,272264889,2892640,366376,38400,astrocyte,1,1,L1astrocyteglial,False,10,23,Layer 1,astrocyte,33,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
39460,272367447,2935032,361944,3840,astrocyte,1,1,L1astrocyteglial,False,10,27,Layer 1,astrocyte,37,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50297,105659742627,1312192,2293887,163977,microglia/opc,7,7,layer-unclassifiedglialmicroglia/opc,False,0,0,0,0,0,1,microglia/opc_layer-unclassifiedglialmicroglia...,True,False,False,None
50298,105671572495,659392,2298496,171138,oligodendrocyte,7,7,layer-unclassifiedglialoligodendrocyte,False,0,0,0,0,0,1,oligodendrocyte_layer-unclassifiedglialoligode...,True,False,False,None
50299,105672884077,974687,2297216,167606,oligodendrocyte,7,7,layer-unclassifiedglialoligodendrocyte,False,0,0,0,0,0,1,oligodendrocyte_layer-unclassifiedglialoligode...,True,False,False,None
50300,105673920419,1223904,2297535,169289,microglia/opc,7,7,layer-unclassifiedglialmicroglia/opc,False,0,0,0,0,0,1,microglia/opc_layer-unclassifiedglialmicroglia...,True,False,False,None


In [44]:
nodes_new.to_csv('cell_positions_h01_full.csv')

In [45]:
nodes_new_sub = nodes_new[(nodes_new['z'] >= 11000) & (nodes_new['z'] <= 121002)] # z >= 11000 preserva o volume da soma && z <= 121002 preserva o tronco apical Exc/Inh.
nodes_new_sub

,agglo_seg,x,y,z,type,layer,linfo,info,neuron,excitatory synapses,inhibitory synapses,cortical layer,cell type,synapses,repetitions,labels,glia,spiny,pyr,EI
39453,258279044,3018304,367192,26880,astrocyte,1,1,L1astrocyteglial,False,18,43,Layer 1,astrocyte,61,2,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
35704,272206514,2888792,369680,42240,astrocyte,1,1,L1glialmicroglia/opc,False,15,23,Layer 1,astrocyte,38,1,astrocyte_L1glialmicroglia/opc_astrocyte,True,False,False,None
35703,272264889,2892640,366376,38400,astrocyte,1,1,L1astrocyteglial,False,10,23,Layer 1,astrocyte,33,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
39457,272500389,2973488,364960,19200,astrocyte,1,1,L1astrocyteglial,False,17,25,Layer 1,astrocyte,42,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
35698,286632262,2891224,396448,23040,astrocyte,1,1,L1astrocyteglial,False,8,33,Layer 1,astrocyte,41,1,astrocyte_L1astrocyteglial_astrocyte,True,False,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31210,78813745791,706696,2214424,116949,oligodendrocyte,7,7,WMglialoligodendrocyte,False,0,1,White matter,oligodendrocyte,1,1,oligodendrocyte_WMglialoligodendrocyte_oligode...,True,False,False,None
38032,78858499231,966776,2228232,116949,oligodendrocyte,7,7,WMglialoligodendrocyte,False,0,0,White matter,oligodendrocyte,0,1,oligodendrocyte_WMglialoligodendrocyte_oligode...,True,False,False,None
33396,79033857222,1132696,2269344,116949,oligodendrocyte,7,7,WMglialoligodendrocyte,False,0,0,White matter,oligodendrocyte,0,1,oligodendrocyte_WMglialoligodendrocyte_oligode...,True,False,False,None
38062,79207564415,893960,2309040,116949,oligodendrocyte,7,7,oligodendrocyte,False,0,0,White matter,oligodendrocyte,0,1,oligodendrocyte_oligodendrocyte_oligodendrocyte,False,False,False,None


In [46]:
# Next
# 1) Remove the edges
# 2) Include everything that could be a neuron
# 3) Use the number of synapses to correct for missing neurons (e.g., n > 100 or n > 200, ...?)
# 4) Set 'neuron == True' and save the spreadsheet